# Inspect training inputs

In [ ]:
from io import BytesIO
from pathlib import Path
from types import SimpleNamespace
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

# Find the repository whether Jupyter starts at the root or under notebooks/.
_here = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_here, *_here.parents) if (p / "training" / "data.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter from this repository or one of its subdirectories.")
sys.path.insert(0, str(REPO_ROOT))

from dataset.preprocessing import decode_foreground_mask
from training.config import prepare_experiment
from training.data import build_dataset, build_loader, model_inputs

# Match the training command here.
EXPERIMENT_PATH = REPO_ROOT / "configs/ablations/b1_glp_corrected_thresholding_softplus.json"
DATA_PATH = REPO_ROOT / "configs/data.json"
SPLIT = "train"

NUM_EXAMPLES = 5
CHECK_SAMPLES = 32
SEED = 7

In [ ]:
experiment, data_config = prepare_experiment(EXPERIMENT_PATH, DATA_PATH)
dataset = build_dataset(
    data_config,
    experiment,
    experiment["model"],
    SPLIT,
    is_train=(SPLIT == "train"),
)

model_spec = SimpleNamespace(
    uses_material=bool(experiment["model"].get("use_material_prior", False)),
    uses_text=bool(experiment["model"].get("use_text", False)),
)

print(f"Experiment: {experiment['name']} | split: {SPLIT} | samples: {len(dataset):,}")
print(
    f"Depth: {experiment['data']['depth']} | "
    f"segmentation: {experiment['data'].get('segmentation', 'none')} | "
    f"points: {dataset.pc_dims}"
)
print(
    f"Material prior: {model_spec.uses_material} | "
    f"text embedding: {model_spec.uses_text}"
)

_IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def reseed(offset=0):
    random.seed(SEED + offset)
    np.random.seed(SEED + offset)
    torch.manual_seed(SEED + offset)


def prepared_input(index):
    """Expose the aligned RGB/mask/depth produced by the dataset's current adapter."""
    asin = dataset.filenames_list[index]
    record = dataset.emma_dataset.get_record(asin)
    image_hash = next(iter(dataset.emma_dataset.get_image_hashes(record)))
    image_bytes = dataset.emma_dataset.get_images(image_hashes=[image_hash])[image_hash]
    image = np.asarray(Image.open(BytesIO(image_bytes)).convert("RGB"))
    mask = (
        decode_foreground_mask(dataset.mask_source[asin])
        if dataset.mask_source is not None
        else None
    )
    return dataset.depth_adapter.prepare(image, mask, dataset.depth_source[asin])


def model_rgb(sample):
    image = sample["image"].squeeze(0).detach().cpu().permute(1, 2, 0).numpy()
    return np.clip(image * _IMAGENET_STD + _IMAGENET_MEAN, 0.0, 1.0)


def point_array(sample):
    return sample["pc_incomplete"].squeeze(0).detach().cpu().numpy()


def material_indices(sample, limit=8):
    prior = np.asarray(sample["material_prior"])
    target = np.asarray(sample["material_target"])
    chosen = set(np.argsort(prior)[-limit:].tolist()) | set(np.flatnonzero(target).tolist())
    chosen = np.array(sorted(chosen, key=lambda i: prior[i]))
    return chosen, prior, target

In [ ]:
# Lightweight health check through the real dataset __getitem__ path.
failures = []
check_count = min(CHECK_SAMPLES, len(dataset))

for index in range(check_count):
    try:
        reseed(index)
        sample = dataset[index]
        points = point_array(sample)
        prior = np.asarray(sample["material_prior"])
        target = np.asarray(sample["material_target"])

        checks = {
            "image shape": tuple(sample["image"].shape) == (1, 3, 224, 224),
            "point-cloud shape": tuple(sample["pc_incomplete"].shape) == (1, dataset.pc_dims, 3),
            "finite image": bool(torch.isfinite(sample["image"]).all()),
            "finite point cloud": bool(np.isfinite(points).all()),
            "non-empty point cloud": bool(np.any(~np.all(np.isclose(points, 0.0), axis=1))),
            "positive mass/volume/density": bool(
                np.isfinite([sample["mass"], sample["volume"], sample["density"]]).all()
                and min(sample["mass"], sample["volume"], sample["density"]) > 0
            ),
            "material vector size": prior.size == target.size == dataset.num_materials,
            "finite material vectors": bool(np.isfinite(prior).all() and np.isfinite(target).all()),
            "target material present": bool(np.count_nonzero(target) > 0),
        }
        if model_spec.uses_text:
            embedding = np.asarray(sample["text_embedding"]) if "text_embedding" in sample else None
            checks["text embedding"] = bool(
                embedding is not None
                and embedding.size == dataset.text_embedding_dims
                and np.isfinite(embedding).all()
            )

        problems = [name for name, passed in checks.items() if not passed]
        if problems:
            failures.append((index, str(sample["filename"]), problems))
    except Exception as exc:
        failures.append((index, str(dataset.filenames_list[index]), [f"{type(exc).__name__}: {exc}"]))

print(f"Passed: {check_count - len(failures)}/{check_count}")
for index, asin, problems in failures:
    print(f"[{index}] {asin}: " + "; ".join(problems))

In [ ]:
# Visualize examples spread across the split.
indices = np.linspace(0, len(dataset) - 1, min(NUM_EXAMPLES, len(dataset)), dtype=int)
examples = []
for index in indices:
    reseed(int(index))
    examples.append((int(index), prepared_input(int(index)), dataset[int(index)]))

fig = plt.figure(figsize=(20, 4.2 * len(examples)), constrained_layout=True)

for row, (index, prepared, sample) in enumerate(examples):
    asin = str(sample["filename"])

    ax = fig.add_subplot(len(examples), 5, row * 5 + 1)
    ax.imshow(prepared.image)
    if np.any(prepared.foreground_mask):
        ax.contour(prepared.foreground_mask, levels=[0.5], colors="lime", linewidths=0.8)
    ax.set_title(f"{asin}\naligned RGB + mask")
    ax.axis("off")

    ax = fig.add_subplot(len(examples), 5, row * 5 + 2)
    depth_view = np.ma.masked_less_equal(prepared.depth, 0)
    depth_image = ax.imshow(depth_view, cmap="magma")
    fig.colorbar(depth_image, ax=ax, fraction=0.046)
    ax.set_title(f"masked depth\n{np.count_nonzero(prepared.depth):,} pixels")
    ax.axis("off")

    ax = fig.add_subplot(len(examples), 5, row * 5 + 3)
    ax.imshow(model_rgb(sample))
    ax.set_title("actual model RGB")
    ax.axis("off")

    ax = fig.add_subplot(len(examples), 5, row * 5 + 4, projection="3d")
    points = point_array(sample)
    points = points[~np.all(np.isclose(points, 0.0), axis=1)]
    if len(points):
        ax.scatter(points[:, 0], points[:, 2], points[:, 1], c=points[:, 2], s=5, cmap="viridis")
        spans = np.ptp(points, axis=0)
        ax.set_box_aspect(np.maximum(spans[[0, 2, 1]], 1e-6))
    ax.set_xlabel("x")
    ax.set_ylabel("z")
    ax.set_zlabel("y")
    ax.set_title(f"sampled cloud\n{len(points):,}/{dataset.pc_dims:,} points")

    ax = fig.add_subplot(len(examples), 5, row * 5 + 5)
    chosen, prior, target = material_indices(sample)
    positions = np.arange(len(chosen))
    labels = [str(dataset.material_names[i]) for i in chosen]
    ax.barh(positions, prior[chosen], label="prior")
    target_positions = positions[target[chosen] > 0]
    ax.scatter(np.ones(len(target_positions)), target_positions, marker="|", s=140, label="target")
    ax.set_yticks(positions, labels)
    ax.set_xlim(0, max(1.02, float(prior[chosen].max()) * 1.05))
    extra = ""
    if "text_embedding" in sample:
        extra = f" | text={np.asarray(sample['text_embedding']).shape[0]}"
    ax.set_title(f"materials | mass={float(sample['mass']):.3g} kg{extra}")
    ax.legend(loc="lower right", fontsize=8)

plt.show()

In [ ]:
# Confirm the current DataLoader fields and model_inputs order used by training.
reseed()
loader = build_loader(
    dataset,
    batch_size=min(4, len(dataset)),
    workers=0,
    device_type="cpu",
    shuffle=False,
    drop_last=False,
    seed=SEED,
)
batch = next(iter(loader))
packed = model_inputs(batch, model_spec, torch.device("cpu"))

packed_names = ["point cloud", "RGB image"]
if model_spec.uses_material:
    packed_names.append("material prior")
if model_spec.uses_text:
    packed_names.append("text embedding")

print("Batch fields:")
for name, value in batch.items():
    shape = tuple(value.shape) if hasattr(value, "shape") else f"{len(value)} values"
    print(f"  {name:20s} {shape}")

print("\nmodel_inputs order:")
for name, value in zip(packed_names, packed):
    print(f"  {name:20s} {tuple(value.shape)}  dtype={value.dtype}")